# imports

In [8]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [9]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [10]:
ASSET = 'BTC'
INTERVAL = '4h'

# Load from Gold Layer

In [11]:
query = f"""
    SELECT * 
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

print(f"Loading {ASSET} {INTERVAL} from gold_crypto_features...")
df = conn.execute(query).df()
print(f"Loaded: {df.shape}")

Loading BTC 4h from gold_crypto_features...
Loaded: (13274, 52)


# datetime index

In [12]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,date,open,high,low,close,volume,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
0,BTC,Crypto,Bybit,4h,2020-04-27 12:00:00,7713.5,7722.5,7624.0,7669.5,1068.971,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
1,BTC,Crypto,Bybit,4h,2020-04-27 16:00:00,7669.5,7718.5,7642.0,7700.5,588.447,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2,BTC,Crypto,Bybit,4h,2020-04-27 20:00:00,7700.5,7777.5,7686.0,7772.5,838.893,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [13]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Index set to datetime. Date range: {df.index.min()} → {df.index.max()}")

Index set to datetime. Date range: 2020-04-27 12:00:00 → 2026-05-18 16:00:00


In [14]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-27 12:00:00,BTC,Crypto,Bybit,4h,7713.5,7722.5,7624.0,7669.5,1068.971,98.5,...,-0.005721,0.012843,0.461929,7713.5,832.093,7736.0,7666.0,8.198473e+06,NaN,NaN
2020-04-27 16:00:00,BTC,Crypto,Bybit,4h,7669.5,7718.5,7642.0,7700.5,588.447,76.5,...,0.004034,0.009934,0.764706,7669.5,1068.971,7722.5,7624.0,4.531336e+06,NaN,NaN
2020-04-27 20:00:00,BTC,Crypto,Bybit,4h,7700.5,7777.5,7686.0,7772.5,838.893,91.5,...,0.009307,0.011772,0.945355,7700.5,588.447,7718.5,7642.0,6.520296e+06,NaN,NaN


In [15]:
print(f"Shape: {df.shape}")
df.info(verbose=True, show_counts=True)

Shape: (13274, 51)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 13274 entries, 2020-04-27 12:00:00 to 2026-05-18 16:00:00
Data columns (total 51 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asset_symbol      13274 non-null  object 
 1   asset_class       13274 non-null  object 
 2   exchange          13274 non-null  object 
 3   interval          13274 non-null  object 
 4   open              13274 non-null  float64
 5   high              13274 non-null  float64
 6   low               13274 non-null  float64
 7   close             13274 non-null  float64
 8   volume            13274 non-null  float64
 9   daily_volatility  13274 non-null  float64
 10  sma_7             13274 non-null  float64
 11  sma_30            13274 non-null  float64
 12  rsi_14            13274 non-null  float64
 13  macd              13274 non-null  float64
 14  macd_signal       13274 non-null  float64
 15  macd_histogram    13274 non-null 

# drop metadata cols

In [16]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 47


# check  specific cols (OI, Turnover, Funding Rate)

In [17]:
new_cols = ['turnover', 'open_interest', 'funding_rate']
available_new = [c for c in new_cols if c in df.columns]
missing_new = [c for c in new_cols if c not in df.columns]

print(f"Columns available: {available_new}")
print(f"Columns MISSING:    {missing_new}")

Columns available: ['turnover', 'open_interest', 'funding_rate']
Columns MISSING:    []
